## Section 10 — Extension Deep Learning
### CNN from Scratch + Transfer Learning (MobileNetV2) + Étude Comparative ML vs DL
---

In [ ]:
print("\n" + "="*80)
print("SECTION 10 - EXTENSION DEEP LEARNING")
print("="*80)

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras                        import layers, models, callbacks
from tensorflow.keras.applications          import MobileNetV2
from tensorflow.keras.preprocessing.image   import ImageDataGenerator
from tensorflow.keras.optimizers            import Adam
from tensorflow.keras.utils                 import to_categorical
from sklearn.metrics                         import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection                 import train_test_split

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU disponible     : {len(tf.config.list_physical_devices('GPU')) > 0}")

# ── Constantes partagées avec le reste du notebook ───────────────────────────
IMG_SIZE      = (224, 224)
NUM_CLASSES   = len(SELECTED_CLASSES)          # 10 classes Tomate
BATCH_SIZE    = 32
EPOCHS_SCRATCH= 30
EPOCHS_FINETUNE = 20

# Palette cohérente avec sections précédentes
palette = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12',
           '#9b59b6', '#1abc9c', '#e67e22', '#34495e', '#c0392b', '#27ae60']

print(f"\nClasses : {NUM_CLASSES}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Images disponibles : {len(df_tomato)}")

### 10.1 — Préparation du dataset pour le Deep Learning

In [ ]:
print("\n" + "-"*80)
print("10.1 - Préparation du Dataset DL (chargement images + augmentation)")
print("-"*80)

# ── Encodage des labels (cohérent avec section 9) ────────────────────────────
# Réutilise le LabelEncoder 'le' de la section 9
y_dl = le.transform(df_tomato['class'].values)   # entiers 0..9

# ── Chargement des images en mémoire ─────────────────────────────────────────
# On charge toutes les images prétraitées via preprocess_image() de la section 5
# → cohérence totale avec le pipeline ML (même resize, même normalisation)
print("\nChargement des images...")
X_imgs  = []
y_imgs  = []
valid_paths = []

for _, row in df_tomato.iterrows():
    try:
        img = cv2.imread(str(row['path']))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
        X_imgs.append(img)
        y_imgs.append(le.transform([row['class']])[0])
        valid_paths.append(row['path'])
    except Exception:
        continue

X_imgs = np.array(X_imgs, dtype=np.float32) / 255.0   # normalisation [0, 1]
y_imgs = np.array(y_imgs)

print(f"Images chargées : {X_imgs.shape}")
print(f"Labels shape    : {y_imgs.shape}")
print(f"Valeurs pixels  : min={X_imgs.min():.2f}  max={X_imgs.max():.2f}")

# ── Split Train / Validation / Test (70 / 15 / 15) ───────────────────────────
# On utilise un split légèrement différent du ML pour avoir un set de validation DL
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_imgs, y_imgs, test_size=0.30, stratify=y_imgs, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp,  y_tmp,  test_size=0.50, stratify=y_tmp,  random_state=42)

print(f"\nSplit DL  :")
print(f"  Train      : {X_tr.shape[0]}  images ({X_tr.shape[0]/len(X_imgs)*100:.1f}%)")
print(f"  Validation : {X_val.shape[0]} images ({X_val.shape[0]/len(X_imgs)*100:.1f}%)")
print(f"  Test       : {X_te.shape[0]}  images ({X_te.shape[0]/len(X_imgs)*100:.1f}%)")

# Labels one-hot pour le CNN scratch
y_tr_oh  = to_categorical(y_tr,  NUM_CLASSES)
y_val_oh = to_categorical(y_val, NUM_CLASSES)
y_te_oh  = to_categorical(y_te,  NUM_CLASSES)

# ── Data Augmentation ─────────────────────────────────────────────────────────
# Augmentation légère : rotation, flip, zoom
# → évite le sur-apprentissage sur un dataset de taille modérée
datagen = ImageDataGenerator(
    rotation_range    = 15,
    width_shift_range = 0.1,
    height_shift_range= 0.1,
    horizontal_flip   = True,
    zoom_range        = 0.1,
    fill_mode         = 'reflect'
)
datagen.fit(X_tr)

# ── Visualisation des augmentations ──────────────────────────────────────────
sample_img = X_tr[0:1]
aug_imgs   = [next(datagen.flow(sample_img, batch_size=1))[0] for _ in range(8)]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes[0, 0].imshow(sample_img[0])
axes[0, 0].set_title("Original", fontsize=9, fontweight='bold')
axes[0, 0].axis('off')
for i, aug in enumerate(aug_imgs):
    row, col = divmod(i + 1, 5)
    axes[row, col].imshow(np.clip(aug, 0, 1))
    axes[row, col].set_title(f"Augmentation {i+1}", fontsize=8)
    axes[row, col].axis('off')
axes[1, 4].axis('off')

plt.suptitle("Data Augmentation : exemples générés", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dl_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Utilité : augmenter artificiellement le dataset → meilleure généralisation")

### 10.2 — CNN from Scratch

In [ ]:
print("\n" + "-"*80)
print("10.2 - CNN from Scratch")
print("-"*80)
print("""
Architecture : 3 blocs Conv → MaxPool → BatchNorm → Dropout
               suivi de couches Dense 256 → 128 → softmax(10)
Idée : construire un réseau simple pour établir une baseline DL
       et comprendre l'impact de chaque couche.
""")

def build_cnn_scratch(input_shape=(224, 224, 3), num_classes=10):
    """
    CNN from scratch — architecture progressive :
      Bloc 1 : 32 filtres  → détecte les bords et textures basiques
      Bloc 2 : 64 filtres  → détecte les motifs de lésions
      Bloc 3 : 128 filtres → détecte les structures complexes
      Dense  : classification sur 10 classes

    BatchNormalization → stabilise l'entraînement
    Dropout(0.4)       → réduit le sur-apprentissage
    GlobalAveragePooling → moins de paramètres que Flatten
    """
    inp = layers.Input(shape=input_shape)

    # ── Bloc 1 : 32 filtres ──────────────────────────────────────────────────
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)           # 224→112
    x = layers.Dropout(0.25)(x)

    # ── Bloc 2 : 64 filtres ──────────────────────────────────────────────────
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)           # 112→56
    x = layers.Dropout(0.30)(x)

    # ── Bloc 3 : 128 filtres ─────────────────────────────────────────────────
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)           # 56→28
    x = layers.Dropout(0.35)(x)

    # ── Tête de classification ────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)        # 28×28×128 → 128
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='CNN_Scratch')
    return model


cnn_scratch = build_cnn_scratch(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)

cnn_scratch.compile(
    optimizer = Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

cnn_scratch.summary()
total_params = cnn_scratch.count_params()
print(f"\nNombre total de paramètres : {total_params:,}")

In [ ]:
print("\nEntraînement CNN from Scratch...")

# ── Callbacks ────────────────────────────────────────────────────────────────
cb_scratch = [
    callbacks.EarlyStopping(
        monitor   = 'val_accuracy',
        patience  = 7,
        restore_best_weights = True,
        verbose   = 1
    ),
    callbacks.ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    )
]

t0 = time.time()
history_scratch = cnn_scratch.fit(
    datagen.flow(X_tr, y_tr_oh, batch_size=BATCH_SIZE),
    steps_per_epoch = len(X_tr) // BATCH_SIZE,
    epochs          = EPOCHS_SCRATCH,
    validation_data = (X_val, y_val_oh),
    callbacks       = cb_scratch,
    verbose         = 1
)
scratch_train_time = time.time() - t0
print(f"\nDurée entraînement : {scratch_train_time:.1f}s")

### 10.3 — Transfer Learning : MobileNetV2 (pretrained ImageNet)

In [ ]:
print("\n" + "-"*80)
print("10.3 - Transfer Learning : MobileNetV2 (ImageNet → Tomate)")
print("-"*80)
print("""
Principe du Transfer Learning :
  MobileNetV2 a été pré-entraîné sur ImageNet (1.2M images, 1000 classes).
  Ses couches conv ont appris à détecter : bords, textures, formes, motifs.
  On remplace uniquement la tête de classification (Dense final)
  et on ré-entraîne sur nos 10 classes de tomate.

  Étape 1 — Feature extraction : base gelée, on entraîne seulement la tête
  Étape 2 — Fine-tuning        : on dégèle les 30 dernières couches
""")

def build_mobilenetv2(input_shape=(224, 224, 3), num_classes=10):
    """
    MobileNetV2 fine-tuné :
      - Base   : MobileNetV2 pré-entraîné ImageNet, couches gelées
      - Tête   : GlobalAveragePooling → Dense(256) → Dropout → Dense(10)
      - Avantage : légère (3.4M params), rapide, bonne accuracy
    """
    base = MobileNetV2(
        weights      = 'imagenet',
        include_top  = False,
        input_shape  = input_shape
    )
    base.trainable = False   # Phase 1 : base gelée

    inp = layers.Input(shape=input_shape)
    # MobileNetV2 attend des pixels dans [-1, 1]
    x   = layers.Rescaling(scale=2.0, offset=-1.0)(inp)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.40)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='MobileNetV2_TL')
    return model, base


mobilenet, base_model = build_mobilenetv2(input_shape=(*IMG_SIZE, 3),
                                           num_classes=NUM_CLASSES)
mobilenet.compile(
    optimizer = Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
print(f"Paramètres entraînables (phase 1) : {mobilenet.count_params():,}")
trainable = sum(tf.size(w).numpy() for w in mobilenet.trainable_weights)
print(f"  dont entraînables : {trainable:,}")

In [ ]:
# ── Phase 1 : Feature Extraction (base gelée) ─────────────────────────────
print("\nPhase 1 — Feature Extraction (base MobileNetV2 gelée)...")

cb_tl_phase1 = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    )
]

t0 = time.time()
history_tl1 = mobilenet.fit(
    datagen.flow(X_tr, y_tr_oh, batch_size=BATCH_SIZE),
    steps_per_epoch = len(X_tr) // BATCH_SIZE,
    epochs          = 10,
    validation_data = (X_val, y_val_oh),
    callbacks       = cb_tl_phase1,
    verbose         = 1
)
print(f"Phase 1 terminée en {time.time()-t0:.1f}s")

# ── Phase 2 : Fine-Tuning (dégel des 30 dernières couches) ────────────────
print("\nPhase 2 — Fine-Tuning (dégel des 30 dernières couches)...")

base_model.trainable = True
# Geler toutes sauf les 30 dernières
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Learning rate réduit pour le fine-tuning
mobilenet.compile(
    optimizer = Adam(learning_rate=1e-4),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

cb_tl_phase2 = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    )
]

t0 = time.time()
history_tl2 = mobilenet.fit(
    datagen.flow(X_tr, y_tr_oh, batch_size=BATCH_SIZE),
    steps_per_epoch = len(X_tr) // BATCH_SIZE,
    epochs          = EPOCHS_FINETUNE,
    validation_data = (X_val, y_val_oh),
    callbacks       = cb_tl_phase2,
    verbose         = 1
)
tl_train_time = time.time() - t0
print(f"Phase 2 terminée en {tl_train_time:.1f}s")

### 10.4 — Courbes d'apprentissage

In [ ]:
print("\n" + "-"*80)
print("10.4 - Courbes d'Apprentissage : CNN Scratch vs MobileNetV2")
print("-"*80)

# Fusionner les deux phases du Transfer Learning
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

hist_tl_merged = merge_histories(history_tl1, history_tl2)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

configs = [
    (history_scratch.history, 'CNN Scratch',   '#e74c3c', '#c0392b'),
    (hist_tl_merged,          'MobileNetV2 TL','#3498db', '#2980b9')
]

for col, (hist, name, c_train, c_val) in enumerate(configs):
    epochs_range = range(1, len(hist['accuracy']) + 1)

    # Accuracy
    axes[0, col].plot(epochs_range, hist['accuracy'],
                      color=c_train, linewidth=2, label='Train')
    axes[0, col].plot(epochs_range, hist['val_accuracy'],
                      color=c_val,   linewidth=2, linestyle='--', label='Validation')
    axes[0, col].set_title(f"Accuracy — {name}", fontweight='bold')
    axes[0, col].set_xlabel("Époque")
    axes[0, col].set_ylabel("Accuracy")
    axes[0, col].set_ylim(0, 1.05)
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)
    # Marquer le meilleur
    best_ep  = np.argmax(hist['val_accuracy'])
    best_acc = hist['val_accuracy'][best_ep]
    axes[0, col].axvline(best_ep + 1, color='green', linestyle=':', alpha=0.7)
    axes[0, col].annotate(f"Best: {best_acc:.3f}",
                          xy=(best_ep + 1, best_acc),
                          xytext=(best_ep + 2, best_acc - 0.05),
                          fontsize=8, color='green')

    # Loss
    axes[1, col].plot(epochs_range, hist['loss'],
                      color=c_train, linewidth=2, label='Train')
    axes[1, col].plot(epochs_range, hist['val_loss'],
                      color=c_val,   linewidth=2, linestyle='--', label='Validation')
    axes[1, col].set_title(f"Loss — {name}", fontweight='bold')
    axes[1, col].set_xlabel("Époque")
    axes[1, col].set_ylabel("Loss")
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Courbes d'Apprentissage : CNN Scratch vs MobileNetV2",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dl_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.5 — Évaluation DL sur le jeu de test

In [ ]:
print("\n" + "-"*80)
print("10.5 - Évaluation des Modèles DL sur le Jeu de Test")
print("-"*80)

dl_results = {}

for name, model_dl in [('CNN Scratch', cnn_scratch), ('MobileNetV2 TL', mobilenet)]:
    print(f"\n  Évaluation {name}...")
    y_prob = model_dl.predict(X_te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    dl_results[name] = {
        'y_pred'   : y_pred,
        'y_prob'   : y_prob,
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1'       : f1,
        'color'    : '#e74c3c' if name == 'CNN Scratch' else '#3498db'
    }
    print(f"    Accuracy  : {acc:.4f}")
    print(f"    Precision : {prec:.4f}")
    print(f"    Recall    : {rec:.4f}")
    print(f"    F1-Score  : {f1:.4f}")

# ── Rapports détaillés ────────────────────────────────────────────────────────
for name, r in dl_results.items():
    print(f"\n{'─'*60}")
    print(f"Rapport détaillé — {name}")
    print('─'*60)
    print(classification_report(
        y_te, r['y_pred'],
        target_names=short_names,
        digits=3
    ))

# ── Matrices de confusion DL ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, (name, r) in zip(axes, dl_results.items()):
    cm_dl      = confusion_matrix(y_te, r['y_pred'])
    cm_dl_norm = cm_dl.astype('float') / cm_dl.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_dl_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"{name}\nAcc={r['accuracy']:.3f}  F1={r['f1']:.3f}",
                 fontsize=11, fontweight='bold', color=r['color'])
    ax.set_xticks(range(len(short_names)))
    ax.set_yticks(range(len(short_names)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(short_names, fontsize=7)
    ax.set_ylabel("Vraie classe")
    ax.set_xlabel("Prédite")
    plt.colorbar(im, ax=ax)
    for i in range(cm_dl_norm.shape[0]):
        for j in range(cm_dl_norm.shape[1]):
            ax.text(j, i, f"{cm_dl_norm[i,j]:.2f}",
                    ha='center', va='center', fontsize=6, color='black')

plt.suptitle("Matrices de Confusion — Modèles DL", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dl_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.6 — Étude Comparative ML vs DL

In [ ]:
print("\n" + "="*80)
print("10.6 - ÉTUDE COMPARATIVE COMPLÈTE : ML vs DL")
print("="*80)

# ── Rassembler tous les modèles ───────────────────────────────────────────────
# ML : résultats de la section 9
# DL : résultats de la section 10
all_models = {
    # ── ML ──────────────────────────────────
    'Random Forest'     : {**results['Random Forest'],      'type': 'ML', 'color': '#3498db'},
    'SVM (RBF)'         : {**results['SVM (RBF)'],           'type': 'ML', 'color': '#2980b9'},
    'KNN (k=7)'         : {**results['KNN (k=7)'],           'type': 'ML', 'color': '#1abc9c'},
    'Gradient Boosting' : {**results['Gradient Boosting'],   'type': 'ML', 'color': '#16a085'},
    # ── DL ──────────────────────────────────
    'CNN Scratch'       : {**dl_results['CNN Scratch'],      'type': 'DL', 'color': '#e74c3c'},
    'MobileNetV2 TL'    : {**dl_results['MobileNetV2 TL'],   'type': 'DL', 'color': '#c0392b'},
}

model_names_all = list(all_models.keys())
colors_all      = [all_models[n]['color'] for n in model_names_all]
types_all       = [all_models[n]['type']  for n in model_names_all]

# ── Graphique 1 : Comparaison des 4 métriques ────────────────────────────────
metrics     = ['accuracy', 'precision', 'recall', 'f1']
metric_lbls = ['Accuracy', 'Précision', 'Rappel', 'F1-Score']

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, metric, label in zip(axes, metrics, metric_lbls):
    vals = [all_models[n][metric] for n in model_names_all]
    bars = ax.bar(model_names_all, vals,
                  color=colors_all, alpha=0.85,
                  edgecolor=['#2c3e50' if t=='DL' else 'white' for t in types_all],
                  linewidth=2)
    ax.set_ylim(0, 1.10)
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold')
    ax.set_xticklabels(model_names_all, rotation=30, ha='right', fontsize=8)
    ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    # Ligne séparation ML / DL
    ax.axvline(3.5, color='black', linestyle='-', linewidth=1.5, alpha=0.4)
    ax.text(1.5, 1.06, 'ML', ha='center', fontsize=10,
            fontweight='bold', color='#2c3e50')
    ax.text(4.5, 1.06, 'DL', ha='center', fontsize=10,
            fontweight='bold', color='#c0392b')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle("Comparaison ML vs DL — 6 Modèles — 10 Classes Tomate",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparative_ml_vs_dl_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Graphique 2 : Radar Chart F1 par classe ───────────────────────────────────
print("\nAnalyse F1 par classe — meilleur ML vs meilleur DL")

best_ml_name = max(
    [n for n in all_models if all_models[n]['type'] == 'ML'],
    key=lambda n: all_models[n]['f1']
)
best_dl_name = max(
    [n for n in all_models if all_models[n]['type'] == 'DL'],
    key=lambda n: all_models[n]['f1']
)

from sklearn.metrics import f1_score as f1_per_class

# F1 par classe pour le meilleur ML
f1_ml = f1_per_class(
    y_test,
    all_models[best_ml_name]['y_pred'],
    average=None, zero_division=0
)
# F1 par classe pour le meilleur DL
f1_dl = f1_per_class(
    y_te,
    all_models[best_dl_name]['y_pred'],
    average=None, zero_division=0
)

x = np.arange(NUM_CLASSES)
width = 0.35

fig, ax = plt.subplots(figsize=(16, 6))
bars_ml = ax.bar(x - width/2, f1_ml, width,
                 label=f"ML — {best_ml_name}",
                 color='#3498db', alpha=0.85)
bars_dl = ax.bar(x + width/2, f1_dl, width,
                 label=f"DL — {best_dl_name}",
                 color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel("F1-Score")
ax.set_ylim(0, 1.15)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Seuil 0.80')
ax.set_title(f"F1-Score par classe : Meilleur ML ({best_ml_name}) vs Meilleur DL ({best_dl_name})",
             fontsize=12, fontweight='bold')
ax.legend()

# Annoter les différences
for i, (v_ml, v_dl) in enumerate(zip(f1_ml, f1_dl)):
    diff = v_dl - v_ml
    color = '#27ae60' if diff > 0 else '#c0392b'
    ax.text(i, max(v_ml, v_dl) + 0.03, f"{diff:+.2f}",
            ha='center', fontsize=7, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('f1_per_class_ml_vs_dl.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMeilleur ML : {best_ml_name}  (F1={all_models[best_ml_name]['f1']:.4f})")
print(f"Meilleur DL : {best_dl_name}  (F1={all_models[best_dl_name]['f1']:.4f})")

In [ ]:
# ── Tableau récapitulatif final ───────────────────────────────────────────────
print("\n" + "="*80)
print("TABLEAU COMPARATIF FINAL — ML vs DL")
print("="*80)
print(f"\n{'Modèle':<22} {'Type':>4} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-"*70)

for name, r in all_models.items():
    marker = " ★" if name in (best_ml_name, best_dl_name) else ""
    print(f"{name:<22} {r['type']:>4} "
          f"{r['accuracy']:>10.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>10.4f} {r['f1']:>10.4f}{marker}")

print("-"*70)

# ── Analyse ───────────────────────────────────────────────────────────────────
best_overall = max(all_models, key=lambda n: all_models[n]['f1'])
gain = all_models[best_dl_name]['f1'] - all_models[best_ml_name]['f1']

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  ANALYSE COMPARATIVE                                            ║
╠══════════════════════════════════════════════════════════════════╣
║  Meilleur ML    : {best_ml_name:<20} F1 = {all_models[best_ml_name]['f1']:.4f}   ║
║  Meilleur DL    : {best_dl_name:<20} F1 = {all_models[best_dl_name]['f1']:.4f}   ║
║  Gain DL vs ML  : {gain:+.4f}                                        ║
║  Meilleur global: {best_overall:<20}                        ║
╠══════════════════════════════════════════════════════════════════╣
║  CONCLUSIONS                                                    ║
║  • ML : pipeline rapide, interprétable, features manuelles     ║
║  • DL : apprend les features automatiquement depuis les pixels ║
║  • Transfer Learning >> CNN Scratch (meilleure généralisation) ║
║  • Pour un dataset > 10k images, DL surpasse toujours ML       ║
╚══════════════════════════════════════════════════════════════════╝
""")

print("="*80)
print("✓ SECTION 10 — DEEP LEARNING ET ÉTUDE COMPARATIVE TERMINÉES")
print("="*80)